In [ ]:
import imageio.v2 as imageio
img_arr = imageio.imread('data/bobby.jpg')
img_arr.shape

In [ ]:
import torch
img = torch.from_numpy(img_arr)
out = img.permute(2,0,1)
out.shape # channel, height, width; C x H x W

In [ ]:
import os
data_dir = "data/image-cats"
png_files = [f for f in os.listdir(data_dir) if f.endswith('.png')]
batch = torch.zeros(len(png_files), 3, 256, 256, dtype=torch.uint8)

In [ ]:
for i, filename in enumerate(png_files):
    img_arr = imageio.imread(os.path.join(data_dir, filename))
    img_t = torch.from_numpy(img_arr)
    img_t = img_t.permute(2, 0, 1)
    img_t = img_t[:3] # same as writing img_t[:3, :, :]; keeps first 3 channels in case there are other channels
    batch[i] = img_t
    


In [ ]:
batch = batch.float()
batch /= 255.0

In [ ]:
n_channels = batch.shape[1]
for c in range(n_channels):
    mean= torch.mean(batch[:,c])
    std = torch.std(batch[:,c])
    batch[:,c] = (batch[:,c] - mean) / std

In [ ]:
x = torch.rand(3, 4, 5)
x , x[:,2]

In [ ]:
import imageio

dir_path = 'data/volumetric-dicom/2-LUNG 3.0  B70f-04083'
vol_arr = imageio.volread(dir_path, 'DICOM')
vol_arr.shape

In [ ]:
vol = torch.from_numpy(vol_arr).float()
vol = torch.unsqueeze(vol, 0)

vol.shape

## Tabular Wine

In [ ]:
import csv
import numpy as np
wine_path = "data/winequality-white.csv"
wineq_numpy = np.loadtxt(wine_path, dtype=np.float32, delimiter=";", skiprows=1)
wineq_numpy

In [ ]:
col_list = next(csv.reader(open(wine_path), delimiter=';'))

wineq_numpy.shape, col_list


In [ ]:
wineq = torch.from_numpy(wineq_numpy)

wineq.shape, wineq.dtype

In [ ]:
data = wineq[:, :-1] # select all rows and cols except last
data, data.shape

In [ ]:
target = wineq[:, -1].long()
target.shape

In [ ]:
target_onehot = torch.zeros(target.shape[0], 10)
target_onehot.scatter_(1, target.unsqueeze(1), 1.0)

random_indices = torch.randint(0, len(target), (5,))
print(target[random_indices])
print(target_onehot[random_indices])


In [ ]:
target_unsqueezed = target.unsqueeze(1)
target_unsqueezed


In [ ]:
data_mean = torch.mean(data, dim=0)
data_mean

In [ ]:
data_var = torch.var(data, dim=0)
data_var

In [ ]:
bad_indexes = target <= 3
bad_indexes.shape, bad_indexes.dtype, bad_indexes.sum()


In [ ]:
bad_data = data[bad_indexes]
bad_data.shape


In [ ]:
bad_data = data[target <= 3]
mid_data = data[(target > 3) & (target < 7)]
good_data = data[target >= 7]

bad_mean = torch.mean(bad_data, dim=0)
mid_mean = torch.mean(mid_data, dim=0)
good_mean = torch.mean(good_data, dim=0)

for i, args in enumerate(zip(col_list, bad_mean, mid_mean, good_mean)):
    print('{:2} {:20} {:6.2f} {:6.2f} {:6.2f}'.format(i, *args))


In [ ]:
total_sulfur_threshold = 141.83
total_sulfur_data = data[:,6]
predicted_indexes = torch.lt(total_sulfur_data, total_sulfur_threshold)

predicted_indexes.shape, predicted_indexes.dtype, predicted_indexes.sum()

In [ ]:
actual_indexes = target > 5

actual_indexes.shape, actual_indexes.dtype, actual_indexes.sum()


In [ ]:
n_matches = torch.sum(actual_indexes & predicted_indexes).item()
n_predicted = torch.sum(predicted_indexes).item()
n_actual = torch.sum(actual_indexes).item()

n_matches, n_matches / n_predicted, n_matches / n_actual

## Time series

In [ ]:
bikes_numpy = np.loadtxt(
    "data/hour-fixed.csv",
    dtype=np.float32,
    delimiter=",",
    skiprows=1,
    converters={1: lambda x: float(x[8:10])}
)
bikes = torch.from_numpy(bikes_numpy)
bikes

In [ ]:
bikes.shape, bikes.stride()

In [ ]:
daily_bikes = bikes.view(-1, 24, bikes.shape[1])
daily_bikes.shape, daily_bikes.stride()

In [ ]:
first_day = bikes[:24].long()
weather_onehot = torch.zeros(first_day.shape[0], 4)
first_day[:,9]


In [ ]:
weather_onehot.scatter_(
    dim=1,
    index=first_day[:,9].unsqueeze(1).long() - 1,
    value=1.0)
